In [0]:
dbutils.widgets.text("input_path", "/Volumes/ridepulse/landing/raw/hvfhv/fhvhv_tripdata_2026-01.parquet", "Full HVFHV parquet path")
dbutils.widgets.text("output_dir", "/Volumes/ridepulse/landing/raw/hvfhv_stream/", "Directory to drop simulated batch files into")
dbutils.widgets.text("batch_size", "100", "Rows per simulated batch")
dbutils.widgets.text("num_batches", "20", "Number of batches to generate (0 = all)")
dbutils.widgets.text("sleep_seconds", "10", "Seconds to wait between batches")

input_path = dbutils.widgets.get("input_path")
output_dir = dbutils.widgets.get("output_dir")
batch_size = int(dbutils.widgets.get("batch_size"))
num_batches = int(dbutils.widgets.get("num_batches"))
sleep_seconds = int(dbutils.widgets.get("sleep_seconds"))


from pyspark.sql import functions as F
from pyspark.sql.window import Window

pickup_order = Window.orderBy("pickup_datetime")

trips = (
    spark.read.parquet(input_path)
    .withColumn("row_num", F.row_number().over(pickup_order) - 1)
    .withColumn("batch_id", (F.col("row_num") / batch_size).cast("int"))
    .drop("row_num")
    .cache()
)

total_batches = trips.select(F.max("batch_id")).first()[0] + 1
batches_to_write = total_batches if num_batches == 0 else min(num_batches, total_batches)
print(f"{total_batches} batches available, writing {batches_to_write}")

import time

for batch_id in range(batches_to_write):
    batch_pdf = trips.filter(F.col("batch_id") == batch_id).drop("batch_id").toPandas()
    out_path = f"{output_dir.rstrip('/')}/batch_{batch_id:03d}.parquet"
    batch_pdf.to_parquet(out_path)
    print(f"Wrote {out_path} ({len(batch_pdf)} rows)")

    if batch_id < batches_to_write - 1:
        time.sleep(sleep_seconds)

print("Done.")
